# 第 41 课：受约束的 LLM 后处理与端到端语音系统

LLM 可以利用长上下文修正标点、抽取结构和总结，但最危险的问题是把“不确定”改成“听起来合理但原音频没说过”。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 后处理与语义 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 40 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 证据转写、受约束 LLM、任务安全 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：证据转写、受约束 LLM、任务安全。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：raw ASR 与 normalized text 的证据边界；置信度；N-best；会话状态隔离。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](40_NLU意图识别_槽位抽取与对话状态.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：带时间、说话人、候选与置信度的可追溯识别结果
  ↓ 本课要学会的变换、状态或判断
输出：保留原证据、可校准、可拒绝或澄清的文本/语义结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

项目根目录: <REPO_ROOT>


## 1. 给 LLM 的输入不应只有 1-best

推荐同时提供：

- stable transcript 与可修改 partial；
- N-best/lattice 关键候选及 acoustic/LM score；
- token/word confidence 和时间戳；
- 允许的词典、实体和 JSON schema；
- 明确规则：不得添加候选中没有证据的数字、人名和否定词。

## 2. 分离 transcript 与 interpretation

```json
{
  "verbatim_transcript": "把会议改到周四三点",
  "normalized_transcript": "把会议改到周四15:00",
  "intent": "reschedule_meeting",
  "slots": {"day": "周四", "time": "15:00"},
  "needs_confirmation": true
}
```

原始转写、规范化文本和语义解释必须分别保存，不能用语义结果覆盖证据。

In [2]:
allowed_intents={"open_ac","close_ac","set_temp","reschedule_meeting","unknown"}
def validate_semantic(result):
    errors=[]
    if result.get("intent") not in allowed_intents:errors.append("unknown intent")
    if "temperature" in result.get("slots",{}) and not 16<=result["slots"]["temperature"]<=30:errors.append("temperature out of range")
    if not result.get("verbatim_transcript"):errors.append("missing evidence transcript")
    return errors
print(validate_semantic({"verbatim_transcript":"温度调到四十度","intent":"set_temp","slots":{"temperature":40}}))

['temperature out of range']


## 3. 完整系统数据流

```text
PCM
 ↓ AEC / Beamforming / NS / AGC
VAD + Streaming Log-Mel
 ↓ Causal/Chunk Encoder + CTC
Prefix Beam / LM / WFST / Hotword
 ↓ partial/stable/final + confidence + timestamps
Punctuation / ITN / Diarization
 ↓ N-best semantic reranking
Intent / Slots / Dialogue State / constrained LLM
 ↓ policy validation / confirmation / action
```

每层都应能旁路、版本化、记录指标并独立回归。

## 4. 端到端验收矩阵

- 前端：SNR、ERLE、VAD miss/FA、endpoint latency；
- ASR：CER/WER、热词、RTF、first/final latency；
- 后处理：标点 F1、ITN accuracy、DER、confidence calibration；
- NLU：intent accuracy、slot F1、task success；
- 安全：错误执行率、需确认召回率、幻觉数字/实体率；
- 系统：P99、并发、断线恢复、资源和成本。

## 最终测试

1. 为什么不能让 LLM 直接覆盖原始 transcript？
2. 哪些词的错误具有特别高风险？
3. semantic module 能否挽救已被 beam 删除的正确候选？
4. 为什么每层要有版本号和旁路？
5. 系统最终指标为什么不只是 WER？

<details><summary>展开参考答案</summary>

1. 会丢失证据并掩盖幻觉。2. 数字、人名、金额、时间、否定词和动作词。3. 不能可靠挽救。4. 为审计、定位、灰度和回滚。5. 用户关心任务成功、延迟、稳定性和错误执行风险。

</details>

## 课程完成

现在路线已经从麦克风前端一直延伸到语义执行与生产部署。下一步应选择一个具体目标场景（命令词、会议转写或实时字幕），用真实数据完成端到端项目，而不是继续堆叠名词。

<!-- course-upgrade-v2 -->
## 强化练习：第 41 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `证据转写`、`受约束 LLM`、`任务安全`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**LLM 把不确定数字改成常见数字**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**校验结构化输出并禁止无证据实体**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**画出从 PCM 到 action 的版本化审计链**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：证据转写、受约束 LLM、任务安全。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 证据转写、受约束 LLM、任务安全。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
